# 並列対戦 → self/相手モデル学習ループ（AdamW state継続 + Cosine decay版）

3600エピソードごとの継続学習で、モデル重みだけでなく各 `cluster × role(self/opponent)` の **AdamW optimizer state、CosineAnnealingLR scheduler state、累積更新回数**も引き継ぎます。

学習率は **`1e-4` から開始し、全200回のモデル更新を通してCosine decayで `1e-5` まで低下**します。学習率は1回の3600エピソード学習中には固定し、その学習が完了したタイミングでschedulerを1 step進めます。

各 `model_episodeXXXX` にはモデルと一緒に `_training_state` のスナップショットも保存します。Notebookを再起動した場合は、最新の `model_episodeXXXX` に対応するstateから復元します。


In [16]:
from __future__ import annotations

from datetime import datetime
import importlib
import os
import re
import shutil
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
import torch
from IPython.display import clear_output, display


def find_match_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / 'tools' / 'run_matches_round_robin.py').exists():
            return candidate
    raise FileNotFoundError('並列対戦リポジトリが見つかりません。')


MATCH_ROOT = find_match_root()

# 既存Notebookと同じ学習worktreeを使用する。
# generated stateful trainerは、このTRAIN_ROOT配下のagents/rl_mctsをimportする。
TRAIN_ROOT = MATCH_ROOT

RUNNER = MATCH_ROOT / 'tools' / 'run_matches_round_robin.py'
PREPROCESSOR = TRAIN_ROOT / 'tools' / 'train' / 'preprocess_match_agents.py'
BASE_TRAIN_SCRIPT = TRAIN_ROOT / 'tools' / 'train' / 'train_imitation.py'
PARALLEL_TRAINER = MATCH_ROOT / 'tools' / 'benchmark_parallel_training.py'

# 元train_imitation.pyは変更せず、Notebook実行時にstateful版を生成する。
STATEFUL_TRAIN_SCRIPT = (
    MATCH_ROOT
    / 'results'
    / '_generated'
    / 'train_imitation_stateful.py'
)
TRAIN_SCRIPT = STATEFUL_TRAIN_SCRIPT

venv_python = MATCH_ROOT / '.venv' / ('Scripts/python.exe' if os.name == 'nt' else 'bin/python')
PYTHON = venv_python if venv_python.exists() else Path(sys.executable)

for required in (PREPROCESSOR, BASE_TRAIN_SCRIPT, PARALLEL_TRAINER):
    if not required.exists():
        raise FileNotFoundError(f'並列学習に必要なスクリプトがありません: {required}')

if str(MATCH_ROOT) not in sys.path:
    sys.path.insert(0, str(MATCH_ROOT))

import tools.parallel_selfplay_training as selfplay_training
importlib.reload(selfplay_training)

run_parallel_games = selfplay_training.run_parallel_games
train_models = selfplay_training.train_models
read_parallel_training_metrics = selfplay_training.read_parallel_training_metrics

AGENT_SOURCE_ROOT = (
    MATCH_ROOT / 'agents' / '16model_pretrained_source'
)
MODEL_SOURCE_ROOT = AGENT_SOURCE_ROOT / 'cluster_00' / 'src'
if str(MODEL_SOURCE_ROOT) not in sys.path:
    sys.path.insert(0, str(MODEL_SOURCE_ROOT))

model_file = MODEL_SOURCE_ROOT / 'rl_mcts' / 'model.py'
if not model_file.exists():
    model_file = MODEL_SOURCE_ROOT / 'model.py'
if not model_file.exists():
    raise FileNotFoundError(f'rl_mcts.model source not found under {MODEL_SOURCE_ROOT}')
os.environ['SELFPLAY_MODEL_SOURCE_ROOT'] = str(MODEL_SOURCE_ROOT)

spec = importlib.util.spec_from_file_location('rl_mcts.model', model_file)
module = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = module
spec.loader.exec_module(module)
create_model = module.create_model

print(f'MATCH_ROOT={MATCH_ROOT}')
print(f'TRAIN_ROOT={TRAIN_ROOT}')
print(f'PYTHON={PYTHON}')
print(f'BASE_TRAIN_SCRIPT={BASE_TRAIN_SCRIPT}')
print(f'STATEFUL_TRAIN_SCRIPT={STATEFUL_TRAIN_SCRIPT}')


MATCH_ROOT=C:\mizuki\PokeTCG\pokemon-tcg-agent
TRAIN_ROOT=C:\mizuki\PokeTCG\pokemon-tcg-agent
PYTHON=C:\mizuki\PokeTCG\pokemon-tcg-agent\.venv\Scripts\python.exe
BASE_TRAIN_SCRIPT=C:\mizuki\PokeTCG\pokemon-tcg-agent\tools\train\train_imitation.py
STATEFUL_TRAIN_SCRIPT=C:\mizuki\PokeTCG\pokemon-tcg-agent\results\_generated\train_imitation_stateful.py


In [17]:
# stateful版 train_imitation.py を生成する。
# 元の pokemon-tcg-agent/tools/train/train_imitation.py は変更しない。

STATEFUL_TRAIN_SCRIPT.parent.mkdir(parents=True, exist_ok=True)

STATEFUL_TRAIN_SOURCE = r"""
from __future__ import annotations

import argparse
import csv
import json
import os
import pickle
import random
import re
import sys
import time
from pathlib import Path
from typing import Iterator

TRAIN_ROOT = Path(os.environ["SELFPLAY_TRAIN_ROOT"]).resolve()
SRC_ROOT = Path(
    os.environ.get(
        "SELFPLAY_MODEL_SOURCE_ROOT",
        TRAIN_ROOT / "agents" / "rl_mcts" / "src",
    )
).resolve()
sys.path.insert(0, str(TRAIN_ROOT))
sys.path.insert(0, str(SRC_ROOT))

import torch
import torch.nn.functional as F

from rl_mcts.mcts import MAX_ACTIONS, LearnInput
from rl_mcts.model import create_model
from tools.value_training import build_value_loss, forward_for_training


def load_shard(path: Path) -> list[tuple]:
    with open(path, "rb") as f:
        return pickle.load(f)


def build_batch_tensors(batch: list[tuple], device: torch.device):
    input_enc = LearnInput()
    input_dec = LearnInput()
    mask: list[float] = []
    label_value: list[float] = []
    chosen_indices: list[int] = []

    for (
        enc_index,
        enc_value,
        enc_offset,
        dec_index,
        dec_value,
        dec_offset,
        chosen_index,
        value,
    ) in batch:
        enc_count = len(input_enc.index)
        input_enc.index.extend(enc_index)
        input_enc.value.extend(enc_value)
        input_enc.offset.extend(o + enc_count for o in enc_offset)

        dec_count = len(input_dec.index)
        input_dec.index.extend(dec_index)
        input_dec.value.extend(dec_value)
        input_dec.offset.extend(o + dec_count for o in dec_offset)

        label_value.append(value)
        chosen_indices.append(chosen_index)

        n_candidates = len(dec_offset)
        mask.extend([1.0] * n_candidates)
        for _ in range(MAX_ACTIONS - n_candidates):
            mask.append(0.0)
            input_dec.offset.append(len(input_dec.index))

    n = len(batch)
    mask_tensor = torch.tensor(
        mask, dtype=torch.float32, device=device
    ).view(n, -1)
    label_value_tensor = torch.tensor(
        label_value, dtype=torch.float32, device=device
    ).view(n, -1)
    chosen_index_tensor = torch.tensor(
        chosen_indices, dtype=torch.long, device=device
    )

    tensors = (
        torch.tensor(input_enc.index, dtype=torch.int32, device=device),
        torch.tensor(input_enc.value, dtype=torch.float32, device=device),
        torch.tensor(input_enc.offset, dtype=torch.int32, device=device),
        torch.tensor(input_dec.index, dtype=torch.int32, device=device),
        torch.tensor(input_dec.value, dtype=torch.float32, device=device),
        torch.tensor(input_dec.offset, dtype=torch.int32, device=device),
    )
    return tensors, mask_tensor, label_value_tensor, chosen_index_tensor


def iter_batches(
    shard_paths: list[Path],
    batch_size: int,
    shuffle: bool,
) -> Iterator[list[tuple]]:
    paths = list(shard_paths)
    if shuffle:
        random.shuffle(paths)

    for path in paths:
        samples = load_shard(path)
        if shuffle:
            random.shuffle(samples)
        for start in range(0, len(samples) - batch_size + 1, batch_size):
            yield samples[start : start + batch_size]


def evaluate(
    model,
    shard_paths: list[Path],
    batch_size: int,
    device: torch.device,
) -> float:
    if not shard_paths:
        return 0.0

    model.eval()
    correct = total = 0

    with torch.no_grad():
        for batch in iter_batches(shard_paths, batch_size, shuffle=False):
            tensors, mask_tensor, _, chosen_index_tensor = build_batch_tensors(
                batch, device
            )
            _, out_dec = model(*tensors)
            masked_logits = out_dec.masked_fill(
                mask_tensor == 0, float("-inf")
            )
            pred = masked_logits.argmax(dim=1)
            correct += int((pred == chosen_index_tensor).sum().item())
            total += len(batch)

    return correct / total if total else 0.0


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser()
    parser.add_argument("--shards", type=Path, required=True)
    parser.add_argument("--val-shards", type=int, default=1)
    parser.add_argument("--epochs", type=int, default=5)
    parser.add_argument("--batch-size", type=int, default=128)
    parser.add_argument("--lr", type=float, default=3e-4)
    parser.add_argument("--initial-model", type=Path, default=None)
    parser.add_argument("--output-model", type=Path, required=True)
    parser.add_argument("--metrics-file", type=Path, required=True)
    parser.add_argument("--seed", type=int, default=None)
    parser.add_argument("--device", default="auto")
    return parser.parse_args()


def select_device(name: str) -> torch.device:
    normalized = name.lower()
    if normalized == "auto":
        if torch.cuda.is_available():
            normalized = "cuda"
        elif torch.backends.mps.is_available():
            normalized = "mps"
        else:
            normalized = "cpu"

    device = torch.device(normalized)

    if device.type == "cuda" and not torch.cuda.is_available():
        raise SystemExit("CUDAを利用できません。")
    if device.type == "mps" and not torch.backends.mps.is_available():
        raise SystemExit("MPSを利用できません。")

    return device


def infer_training_identity(output_model: Path) -> tuple[str, str]:
    match = re.search(r"cluster_\d+", str(output_model))
    if not match:
        raise RuntimeError(
            f"output-modelからcluster名を判定できません: {output_model}"
        )

    cluster = match.group(0)
    role = "opponent" if output_model.name == "opponent_model.pth" else "self"
    return cluster, role


def atomic_torch_save(obj, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(
        f".{path.name}.{os.getpid()}.tmp"
    )
    try:
        torch.save(obj, temporary)
        os.replace(temporary, path)
    finally:
        temporary.unlink(missing_ok=True)


def main() -> None:
    args = parse_args()

    if args.seed is not None:
        random.seed(args.seed)
        torch.manual_seed(args.seed)

    device = select_device(args.device)
    print(f"device: {device}")

    shard_paths = sorted(args.shards.glob("shard_*.pkl"))
    if not shard_paths:
        raise SystemExit(
            f"シャードが見つかりません: {args.shards}"
        )

    manifest_path = args.shards / "manifest.json"
    if manifest_path.exists():
        print(
            "manifest: "
            f"{json.loads(manifest_path.read_text(encoding='utf-8'))}"
        )

    random.shuffle(shard_paths)
    val_count = min(
        max(args.val_shards, 0),
        max(len(shard_paths) - 1, 0),
    )
    val_shards = shard_paths[:val_count]
    train_shards = shard_paths[val_count:]

    print(
        f"train shards={len(train_shards)} "
        f"val shards={len(val_shards)}"
    )

    model = create_model().to(device)
    if args.initial_model and args.initial_model.exists():
        model.load_state_dict(
            torch.load(args.initial_model, map_location=device)
        )
        print(f"loaded initial weights: {args.initial_model}")

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=args.lr,
    )

    # 3600エピソードごとの「モデル更新」単位でCosine decayする。
    # 全学習更新を通して base lr -> min lr へ滑らかに低下させる。
    min_lr = float(
        os.environ.get("SELFPLAY_MIN_LR", "1e-5")
    )
    scheduler_updates = int(
        os.environ.get("SELFPLAY_LR_T_MAX", "200")
    )

    if args.lr <= 0:
        raise ValueError("--lrは正である必要があります。")
    if min_lr <= 0 or min_lr > args.lr:
        raise ValueError(
            f"MIN_LRは0 < MIN_LR <= base lrである必要があります: "
            f"min_lr={min_lr}, base_lr={args.lr}"
        )
    if scheduler_updates <= 0:
        raise ValueError(
            f"Cosine schedulerのT_maxは正である必要があります: "
            f"{scheduler_updates}"
        )

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=scheduler_updates,
        eta_min=min_lr,
    )

    state_root = Path(
        os.environ["SELFPLAY_TRAINING_STATE_ROOT"]
    ).resolve()

    cluster, role = infer_training_identity(args.output_model)
    state_path = (
        state_root
        / cluster
        / role
        / "training_state.pth"
    )

    global_step = 0

    # 前世代のAdamW + scheduler + global_stepを復元する。
    if state_path.exists():
        state = torch.load(
            state_path,
            map_location=device,
        )

        saved_base_lr = float(state.get("base_lr", args.lr))
        saved_min_lr = float(state.get("min_lr", min_lr))
        saved_scheduler_updates = int(
            state.get("scheduler_updates", scheduler_updates)
        )

        if abs(saved_base_lr - args.lr) > 1e-15:
            raise RuntimeError(
                "保存済みoptimizer stateと今回の--lrが一致しません: "
                f"saved={saved_base_lr}, current={args.lr}"
            )
        if abs(saved_min_lr - min_lr) > 1e-15:
            raise RuntimeError(
                "保存済みscheduler stateと今回のmin_lrが一致しません: "
                f"saved={saved_min_lr}, current={min_lr}"
            )
        if saved_scheduler_updates != scheduler_updates:
            raise RuntimeError(
                "保存済みscheduler stateと今回のT_maxが一致しません: "
                f"saved={saved_scheduler_updates}, current={scheduler_updates}"
            )

        optimizer.load_state_dict(state["optimizer"])
        scheduler.load_state_dict(state["scheduler"])
        global_step = int(
            state.get("global_step", scheduler.last_epoch)
        )

        print(f"loaded training state: {state_path}")
        print(
            f"global_step={global_step} "
            f"lr={optimizer.param_groups[0]['lr']:.10g}"
        )
    else:
        print(f"new training state: {state_path}")
        print(
            f"global_step=0 "
            f"lr={optimizer.param_groups[0]['lr']:.10g}"
        )

    args.output_model.parent.mkdir(
        parents=True, exist_ok=True
    )
    args.metrics_file.parent.mkdir(
        parents=True, exist_ok=True
    )

    fieldnames = [
        "epoch",
        "batches",
        "loss",
        "loss_value",
        "loss_policy",
        "train_accuracy",
        "val_accuracy",
        "learning_rate",
        "global_step",
        "elapsed_seconds",
    ]

    with open(
        args.metrics_file,
        "w",
        newline="",
        encoding="utf-8",
    ) as f:
        csv.DictWriter(
            f, fieldnames=fieldnames
        ).writeheader()

    value_loss_kind = os.environ.get("SELFPLAY_VALUE_LOSS", "bce")
    loss_fn_value = build_value_loss(torch, kind=value_loss_kind)

    t0 = time.time()

    for epoch in range(args.epochs):
        model.train()

        batch_count = 0
        total_loss = 0.0
        total_loss_value = 0.0
        total_loss_policy = 0.0
        total_correct = 0
        total_seen = 0

        for batch in iter_batches(
            train_shards,
            args.batch_size,
            shuffle=True,
        ):
            (
                tensors,
                mask_tensor,
                label_value_tensor,
                chosen_index_tensor,
            ) = build_batch_tensors(batch, device)

            optimizer.zero_grad()
            out_enc, out_dec = forward_for_training(model, *tensors)

            loss_value = loss_fn_value(
                out_enc,
                label_value_tensor,
            )

            masked_logits = out_dec.masked_fill(
                mask_tensor == 0,
                float("-inf"),
            )

            loss_policy = F.cross_entropy(
                masked_logits,
                chosen_index_tensor,
            )

            loss = loss_value + loss_policy
            loss.backward()
            optimizer.step()

            # global_stepはoptimizer.step()の累積回数として記録する。
            # 学習率scheduler自体は3600エピソード分の学習が完了した後に
            # 1回だけ進める。
            global_step += 1

            with torch.no_grad():
                pred = masked_logits.argmax(dim=1)
                total_correct += int(
                    (
                        pred
                        == chosen_index_tensor
                    ).sum().item()
                )
                total_seen += len(batch)

            total_loss += float(loss.item())
            total_loss_value += float(
                loss_value.item()
            )
            total_loss_policy += float(
                loss_policy.item()
            )
            batch_count += 1

        val_acc = evaluate(
            model,
            val_shards,
            args.batch_size,
            device,
        )

        elapsed = time.time() - t0

        stats = {
            "epoch": epoch,
            "batches": batch_count,
            "loss": (
                total_loss / batch_count
                if batch_count
                else 0.0
            ),
            "loss_value": (
                total_loss_value / batch_count
                if batch_count
                else 0.0
            ),
            "loss_policy": (
                total_loss_policy / batch_count
                if batch_count
                else 0.0
            ),
            "train_accuracy": (
                total_correct / total_seen
                if total_seen
                else 0.0
            ),
            "val_accuracy": val_acc,
            "learning_rate": optimizer.param_groups[0]["lr"],
            "global_step": global_step,
            "elapsed_seconds": elapsed,
        }

        print(
            f"epoch={epoch} "
            f"loss={stats['loss']:.4f} "
            f"loss_value={stats['loss_value']:.4f} "
            f"loss_policy={stats['loss_policy']:.4f} "
            f"train_acc={stats['train_accuracy']:.3f} "
            f"val_acc={stats['val_accuracy']:.3f} "
            f"lr={stats['learning_rate']:.10g} "
            f"global_step={global_step} "
            f"elapsed={elapsed:.1f}s"
        )

        with open(
            args.metrics_file,
            "a",
            newline="",
            encoding="utf-8",
        ) as f:
            csv.DictWriter(
                f, fieldnames=fieldnames
            ).writerow(stats)

        # model.pthは従来どおり純粋なmodel.state_dict()のままにする。
        atomic_torch_save(
            model.state_dict(),
            args.output_model,
        )

        # AdamW、scheduler、累積stepを別ファイルに保存する。
        # model/optimizer stateは各epoch終了時点で保存する。
        # TRAIN_EPOCHSは現在1だが、複数epochへ変更してもoptimizer stateは失わない。
        atomic_torch_save(
            {
                "optimizer": optimizer.state_dict(),
                "scheduler": scheduler.state_dict(),
                "global_step": global_step,
                "base_lr": args.lr,
                "min_lr": min_lr,
                "scheduler_updates": scheduler_updates,
            },
            state_path,
        )

    # 1回の3600エピソード学習を1 scheduler stepとして扱う。
    # 次のmodel_episodeXXXXは、この更新後の学習率から開始する。
    scheduler.step()

    atomic_torch_save(
        {
            "optimizer": optimizer.state_dict(),
            "scheduler": scheduler.state_dict(),
            "global_step": global_step,
            "base_lr": args.lr,
            "min_lr": min_lr,
            "scheduler_updates": scheduler_updates,
        },
        state_path,
    )

    print(
        f"next learning rate: "
        f"{optimizer.param_groups[0]['lr']:.10g}"
    )

    print(f"saved model: {args.output_model}")
    print(f"saved training state: {state_path}")


if __name__ == "__main__":
    main()
"""

STATEFUL_TRAIN_SCRIPT.write_text(
    STATEFUL_TRAIN_SOURCE,
    encoding='utf-8',
)

print(f'generated: {STATEFUL_TRAIN_SCRIPT}')
print(f'using TRAIN_SCRIPT={TRAIN_SCRIPT}')


generated: C:\mizuki\PokeTCG\pokemon-tcg-agent\results\_generated\train_imitation_stateful.py
using TRAIN_SCRIPT=C:\mizuki\PokeTCG\pokemon-tcg-agent\results\_generated\train_imitation_stateful.py


## 設定

初期パラメータは `gen_000/agents/clXX/{self.pth,opp.pth}` から読み込み、対応する `deck.csv` とともに学習用agentへ配置します。学習後の最新パラメータと世代別checkpointは `agents/16model_pretrained_vtrain/` に保存します。既存の学習用パラメータがある場合は継続学習として扱い、初期パラメータでは上書きしません。

学習率は **`LEARNING_RATE = 1e-4`** から開始し、`TRAINING_UPDATES = 200` 回のモデル更新全体を `CosineAnnealingLR` で減衰させ、最終的に **`MIN_LEARNING_RATE = 1e-5`** へ到達させます。AdamWとschedulerのstateは世代間で継続します。


In [18]:
INITIAL_PARAMETERS_ROOT = MATCH_ROOT / 'gen_000' / 'agents'
TRAINING_AGENTS_ROOT = (
    MATCH_ROOT / 'agents' / '16model_pretrained_vtrain'
)


def prepare_training_agent(agent_index: int) -> None:
    cluster_name = f'cluster_{agent_index:02d}'
    source_agent_dir = AGENT_SOURCE_ROOT / cluster_name
    training_agent_dir = TRAINING_AGENTS_ROOT / cluster_name
    initial_agent_dir = (
        INITIAL_PARAMETERS_ROOT / f'cl{agent_index:02d}'
    )

    if not source_agent_dir.is_dir():
        raise FileNotFoundError(
            f'agentソースがありません: {source_agent_dir}'
        )

    if not training_agent_dir.exists():
        shutil.copytree(
            source_agent_dir,
            training_agent_dir,
            ignore=shutil.ignore_patterns(
                'model.pth',
                'opponent_model.pth',
            ),
        )

    initial_deck = initial_agent_dir / 'deck.csv'
    if not initial_deck.is_file():
        raise FileNotFoundError(
            f'初期パラメータ用deckがありません: {initial_deck}'
        )
    shutil.copy2(
        initial_deck,
        training_agent_dir / 'src' / 'deck.csv',
    )


for index in range(16):
    prepare_training_agent(index)

AGENTS = {
    f'cluster_{index:02d}': (
        TRAINING_AGENTS_ROOT / f'cluster_{index:02d}' / 'src' / 'main.py'
    )
    for index in range(16)
}

SEED = 0


def ensure_model_file(
    model_path: Path,
    initial_model_path: Path,
) -> str:
    if model_path.is_file():
        return 'existing'
    if model_path.exists():
        raise FileExistsError(
            f'モデル保存先がファイルではありません: {model_path}'
        )
    if not initial_model_path.is_file():
        raise FileNotFoundError(
            f'初期パラメータがありません: {initial_model_path}'
        )

    model_path.parent.mkdir(parents=True, exist_ok=True)

    temporary = model_path.with_name(
        f'.{model_path.name}.{os.getpid()}.tmp'
    )
    try:
        try:
            payload = torch.load(
                initial_model_path,
                map_location='cpu',
                weights_only=True,
            )
        except TypeError:
            payload = torch.load(
                initial_model_path,
                map_location='cpu',
            )
        state_dict = (
            payload['state_dict']
            if isinstance(payload, dict) and 'state_dict' in payload
            else payload
        )
        torch.save(state_dict, temporary)
        os.replace(
            temporary,
            model_path,
        )
    finally:
        temporary.unlink(missing_ok=True)

    return f'initialized-from:{initial_model_path}'


INITIAL_WEIGHT_STATUS = {}

for agent_name, main_path in AGENTS.items():
    for required in (
        main_path,
        main_path.parent / 'deck.csv',
    ):
        if not required.is_file():
            raise FileNotFoundError(
                f'{agent_name}のagentファイルがありません: {required}'
            )

    agent_index = int(agent_name.rsplit('_', 1)[1])
    initial_agent_dir = (
        INITIAL_PARAMETERS_ROOT / f'cl{agent_index:02d}'
    )

    INITIAL_WEIGHT_STATUS[agent_name] = {
        'self': ensure_model_file(
            main_path.parent / 'model.pth',
            initial_agent_dir / 'self.pth',
        ),
        'opponent': ensure_model_file(
            main_path.parent / 'opponent_model.pth',
            initial_agent_dir / 'opp.pth',
        ),
    }


CPU_THREADS = os.cpu_count() or 1

GAME_WORKERS = max(
    1,
    CPU_THREADS - 1,
)

TRAINING_WORKERS = max(
    1,
    CPU_THREADS - 1,
)

LANES_PER_WORKER = 400

MATCH_BATCH_SIZE = 256
SEARCH_COUNT = 10
MAX_TURNS = 100
MAX_SELECTIONS = 500
MAX_MATCH_ATTEMPTS = 5
# CUDA、MPS、CPUの順に、PyTorchで利用可能なdeviceを自動選択する。
MATCH_DEVICE = 'auto'

TRAINING_UPDATES = 13
TRAIN_EPOCHS = 1
TRAIN_MINIBATCH_SIZE = 128

# AdamWの基準学習率。
# 既存モデルを継続学習するため、3e-4より保守的な1e-4から開始する。
LEARNING_RATE = 1e-4

# Cosine decayの下限。
MIN_LEARNING_RATE = 1e-5

SHARD_SIZE = 20_000
TRAIN_DEVICE = 'auto'
TRAIN_DEVICE_WAVE_SIZE = TRAINING_WORKERS
LOSS_POLL_INTERVAL_SECONDS = 0.5

RUN_ID = datetime.now().strftime(
    '%Y%m%d_%H%M%S_%f'
)

RUN_ROOT = (
    MATCH_ROOT
    / 'results'
    / 'training_loop'
    / RUN_ID
)

EPISODES_PER_UPDATE = (
    GAME_WORKERS
    * LANES_PER_WORKER
)

MODEL_EPISODE_PATTERN = re.compile(
    r'model_episode(\d+)'
)

START_EPISODES = max(
    (
        int(match.group(1))
        for path in TRAINING_AGENTS_ROOT.iterdir()
        if (
            path.is_dir()
            and (
                match
                := MODEL_EPISODE_PATTERN.fullmatch(
                    path.name
                )
            )
        )
    ),
    default=0,
)

TOTAL_EPISODES = (
    START_EPISODES
    + EPISODES_PER_UPDATE
    * TRAINING_UPDATES
)

# ============================================================
# AdamW / schedulerの「現在値」を置く場所
# ============================================================

TRAINING_STATE_ROOT = (
    TRAINING_AGENTS_ROOT
    / '_training_state'
)

# Notebook再起動時は、最新のmodel_episodeXXXXに保存してある
# stateスナップショットを現在stateへ戻す。
if START_EPISODES > 0:
    state_snapshot = (
        TRAINING_AGENTS_ROOT
        / f'model_episode{START_EPISODES}'
        / '_training_state'
    )

    if state_snapshot.is_dir():
        if TRAINING_STATE_ROOT.exists():
            shutil.rmtree(
                TRAINING_STATE_ROOT
            )

        shutil.copytree(
            state_snapshot,
            TRAINING_STATE_ROOT,
        )

        print(
            'restored optimizer/scheduler state: '
            f'{state_snapshot}'
        )
    else:
        # 旧Notebookで作ったcheckpointにはstateがないため、
        # その場合だけAdamWはこの世代から新規開始になる。
        if TRAINING_STATE_ROOT.exists():
            shutil.rmtree(
                TRAINING_STATE_ROOT
            )

        print(
            'WARNING: '
            f'{state_snapshot} がありません。'
            'モデル重みは継続しますが、AdamW/scheduler stateは'
            'この実行から新規開始します。'
        )
else:
    # model_episodeが1つもないのに古いstateだけ残っていた場合、
    # ランダム初期モデルと混ざらないよう削除する。
    if TRAINING_STATE_ROOT.exists():
        shutil.rmtree(
            TRAINING_STATE_ROOT
        )

TRAINING_STATE_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

# generated stateful trainerへ設定を渡す。
# subprocessはNotebookプロセスの環境変数を継承する。
os.environ[
    'SELFPLAY_TRAINING_STATE_ROOT'
] = str(
    TRAINING_STATE_ROOT
)

os.environ[
    'SELFPLAY_TRAIN_ROOT'
] = str(
    TRAIN_ROOT
)

# Valueは学習時にtanh前のlogitへBCEを掛ける。
os.environ['SELFPLAY_VALUE_LOSS'] = 'bce'
os.environ['SELFPLAY_VALUE_TRAINING_SPACE'] = 'logit'

os.environ[
    'SELFPLAY_LR_T_MAX'
] = str(
    TRAINING_UPDATES
)

os.environ[
    'SELFPLAY_MIN_LR'
] = str(
    MIN_LEARNING_RATE
)

print(
    f'start_episodes={START_EPISODES}, '
    f'final_episodes={TOTAL_EPISODES}, '
    f'new_episodes={TOTAL_EPISODES - START_EPISODES}, '
    f'updates={TRAINING_UPDATES}, '
    f'train_epochs/update={TRAIN_EPOCHS}, '
    f'agents={len(AGENTS)}, '
    f'game_workers={GAME_WORKERS}, '
    f'lanes/worker={LANES_PER_WORKER}, '
    f'episodes/update={EPISODES_PER_UPDATE}, '
    f'training_workers={TRAINING_WORKERS}, '
    f'wave={TRAIN_DEVICE_WAVE_SIZE}, '
    f'train_minibatch={TRAIN_MINIBATCH_SIZE}, '
    f'base_lr={LEARNING_RATE}, '
    f'cosine_t_max={TRAINING_UPDATES}, '
    f'min_lr={MIN_LEARNING_RATE}, '
    f'training_state={TRAINING_STATE_ROOT}, '
    f'initial_parameters={INITIAL_PARAMETERS_ROOT}, '
    f'training_agents={TRAINING_AGENTS_ROOT}, '
    f'output={RUN_ROOT}'
)


start_episodes=0, final_episodes=78000, new_episodes=78000, updates=13, train_epochs/update=1, agents=16, game_workers=15, lanes/worker=400, episodes/update=6000, training_workers=15, wave=15, train_minibatch=128, base_lr=0.0001, cosine_t_max=13, min_lr=1e-05, training_state=C:\mizuki\PokeTCG\pokemon-tcg-agent\agents\16model_pretrained_vtrain\_training_state, initial_parameters=C:\mizuki\PokeTCG\pokemon-tcg-agent\gen_000\agents, training_agents=C:\mizuki\PokeTCG\pokemon-tcg-agent\agents\16model_pretrained_vtrain, output=C:\mizuki\PokeTCG\pokemon-tcg-agent\results\training_loop\20260816_212154_561093


In [19]:
LOSS_HISTORY: dict[tuple[int, str, int], dict] = {}
LOSS_HISTORY_PATH = RUN_ROOT / 'loss_history.csv'


def remember_loss_metrics(
    metrics: list[dict],
    completed_episodes: int,
) -> None:
    global_update = (
        completed_episodes
        // EPISODES_PER_UPDATE
    )

    for metric in metrics:
        row = dict(metric)
        row['update'] = global_update
        row['completed_episodes'] = completed_episodes
        row['global_epoch'] = (
            (global_update - 1)
            * TRAIN_EPOCHS
            + int(row['epoch'])
            + 1
        )

        key = (
            completed_episodes,
            str(row['job']),
            int(row['epoch']),
        )

        LOSS_HISTORY[key] = row


def load_previous_loss_history(
    max_episodes: int,
) -> None:
    latest_episode_dirs: dict[
        int,
        Path,
    ] = {}

    history_root = (
        MATCH_ROOT
        / 'results'
        / 'training_loop'
    )

    if not history_root.is_dir():
        return

    for episode_dir in history_root.glob(
        '*/episode*'
    ):
        match = re.fullmatch(
            r'episode(\d+)',
            episode_dir.name,
        )

        if not match:
            continue

        completed_episodes = int(
            match.group(1)
        )

        metrics_root = (
            episode_dir
            / 'learning'
            / 'parallel_training'
        )

        if (
            completed_episodes > max_episodes
            or not any(
                metrics_root.glob(
                    'jobs/*/metrics.csv'
                )
            )
        ):
            continue

        previous = latest_episode_dirs.get(
            completed_episodes
        )

        if (
            previous is None
            or episode_dir.parent.name
            > previous.parent.name
        ):
            latest_episode_dirs[
                completed_episodes
            ] = episode_dir

    for (
        completed_episodes,
        episode_dir,
    ) in sorted(
        latest_episode_dirs.items()
    ):
        metrics_root = (
            episode_dir
            / 'learning'
            / 'parallel_training'
        )

        remember_loss_metrics(
            read_parallel_training_metrics(
                metrics_root
            ),
            completed_episodes,
        )


def save_loss_history() -> pd.DataFrame:
    frame = (
        pd.DataFrame(
            LOSS_HISTORY.values()
        )
        .sort_values(
            [
                'completed_episodes',
                'epoch',
                'job',
            ]
        )
        .reset_index(
            drop=True
        )
    )

    LOSS_HISTORY_PATH.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    frame.to_csv(
        LOSS_HISTORY_PATH,
        index=False,
    )

    return frame


def draw_live_loss(
    metrics: list[dict],
    completed_episodes: int,
) -> None:
    remember_loss_metrics(
        metrics,
        completed_episodes,
    )

    clear_output(
        wait=True
    )

    if not LOSS_HISTORY:
        print(
            '学習プロセスを実行中です。最初のlossは、'
            'いずれかのモデルが1 epochを完了すると表示されます。'
        )
        return

    frame = save_loss_history()

    aggregate_columns = [
        'loss',
        'loss_value',
        'loss_policy',
        'train_accuracy',
    ]

    if 'learning_rate' in frame.columns:
        aggregate_columns.append(
            'learning_rate'
        )

    averages = (
        frame.groupby(
            [
                'global_epoch',
                'completed_episodes',
            ],
            as_index=False,
        )[aggregate_columns]
        .mean()
    )

    current = frame[
        frame['completed_episodes']
        == completed_episodes
    ]

    if 'learning_rate' in averages.columns:
        fig, axes = plt.subplots(
            1,
            3,
            figsize=(17, 4),
        )
    else:
        fig, axes = plt.subplots(
            1,
            2,
            figsize=(13, 4),
        )

    steps = averages[
        'completed_episodes'
    ]

    axes[0].plot(
        steps,
        averages['loss'],
        marker='o',
        label='total',
    )

    axes[0].plot(
        steps,
        averages['loss_value'],
        marker='o',
        label='value',
    )

    axes[0].plot(
        steps,
        averages['loss_policy'],
        marker='o',
        label='policy',
    )

    axes[0].set(
        title='Mean loss across all updates',
        xlabel='completed games',
        ylabel='loss',
    )

    axes[0].grid(
        alpha=0.3
    )

    axes[0].legend()

    axes[1].plot(
        steps,
        averages['train_accuracy'],
        marker='o',
    )

    axes[1].set(
        title='Mean training accuracy across all updates',
        xlabel='completed games',
        ylabel='accuracy',
        ylim=(0, 1),
    )

    axes[1].grid(
        alpha=0.3
    )

    if 'learning_rate' in averages.columns:
        axes[2].plot(
            steps,
            averages['learning_rate'],
            marker='o',
        )

        axes[2].set(
            title='Mean learning rate',
            xlabel='completed games',
            ylabel='learning rate',
        )

        axes[2].grid(
            alpha=0.3
        )

    expected_current = (
        len(AGENTS)
        * 2
        * TRAIN_EPOCHS
    )

    latest_mean_loss = (
        current.loss.mean()
        if not current.empty
        else averages.loss.iloc[-1]
    )

    extra = ''

    if (
        not current.empty
        and 'learning_rate'
        in current.columns
    ):
        extra = (
            f", mean lr="
            f"{current['learning_rate'].mean():.3g}"
        )

    fig.suptitle(
        f'Games {completed_episodes}/{TOTAL_EPISODES}, '
        f'training update '
        f'{(completed_episodes - START_EPISODES) // EPISODES_PER_UPDATE}'
        f'/{TRAINING_UPDATES}: '
        f'{len(current)}/{expected_current} job-epochs reported, '
        f'cumulative={len(frame)}, '
        f'latest mean loss={latest_mean_loss:.4f}'
        f'{extra}'
    )

    fig.tight_layout()
    display(fig)
    plt.close(fig)


load_previous_loss_history(
    START_EPISODES
)


def run_episode_batch(
    completed_episodes: int,
    update_index: int,
) -> dict:
    episode_root = (
        RUN_ROOT
        / f'episode{completed_episodes}'
    )

    episodes_dir = (
        episode_root
        / 'episodes'
    )

    learning_dir = (
        episode_root
        / 'learning'
    )

    checkpoint_dir = (
        TRAINING_AGENTS_ROOT
        / f'model_episode{completed_episodes}'
    )

    episode_count = run_parallel_games(
        python=PYTHON,
        runner=RUNNER,
        match_root=MATCH_ROOT,
        agents=AGENTS,
        episodes_dir=episodes_dir,
        workers=GAME_WORKERS,
        lanes_per_worker=LANES_PER_WORKER,
        include_self=True,
        match_batch_size=MATCH_BATCH_SIZE,
        search_count=SEARCH_COUNT,
        device=MATCH_DEVICE,
        seed=SEED + update_index,
        max_turns=MAX_TURNS,
        max_selections=MAX_SELECTIONS,
        max_attempts=MAX_MATCH_ATTEMPTS,
    )

    def record_training_progress(
        metrics: list[dict],
    ) -> None:
        draw_live_loss(
            metrics,
            completed_episodes,
        )

    training_summary = train_models(
        python=PYTHON,
        preprocessor=PREPROCESSOR,
        train_script=TRAIN_SCRIPT,
        parallel_trainer=PARALLEL_TRAINER,
        train_root=TRAIN_ROOT,
        match_root=MATCH_ROOT,
        agents=AGENTS,
        episodes_dir=episodes_dir,
        work_dir=learning_dir,
        checkpoint_dir=checkpoint_dir,
        epochs=TRAIN_EPOCHS,
        train_minibatch_size=TRAIN_MINIBATCH_SIZE,
        learning_rate=LEARNING_RATE,
        shard_size=SHARD_SIZE,
        device=TRAIN_DEVICE,
        seed=SEED + update_index * 100,
        training_workers=TRAINING_WORKERS,
        device_wave_size=TRAIN_DEVICE_WAVE_SIZE,
        progress_callback=record_training_progress,
        poll_interval_seconds=LOSS_POLL_INTERVAL_SECONDS,
    )

    # ========================================================
    # その世代のAdamW/scheduler/global_stepもcheckpointへ保存
    # ========================================================

    state_snapshot = (
        checkpoint_dir
        / '_training_state'
    )

    if state_snapshot.exists():
        shutil.rmtree(
            state_snapshot
        )

    shutil.copytree(
        TRAINING_STATE_ROOT,
        state_snapshot,
    )

    return {
        'completed_episodes': completed_episodes,
        'batch_episodes': episode_count,
        'output': str(episode_root),
        'checkpoint': str(checkpoint_dir),
        'training_state': str(state_snapshot),
        'training_seconds': training_summary[
            'trainingSeconds'
        ],
        'training_mode': training_summary[
            'executionMode'
        ],
        'initial_weights': str(
            INITIAL_PARAMETERS_ROOT
        ),
    }


def run_training_loop(
    total_episodes: int = TOTAL_EPISODES,
) -> pd.DataFrame:
    new_episode_count = (
        total_episodes
        - START_EPISODES
    )

    if (
        new_episode_count < 0
        or new_episode_count
        % EPISODES_PER_UPDATE
        != 0
    ):
        raise ValueError(
            f'total_episodes - start_episodes='
            f'{new_episode_count}は'
            f'episodes_per_update='
            f'{EPISODES_PER_UPDATE}'
            f'の倍数にしてください。'
        )

    RUN_ROOT.mkdir(
        parents=True,
        exist_ok=False,
    )

    if LOSS_HISTORY:
        draw_live_loss(
            [],
            START_EPISODES,
        )

    rows = []

    for (
        update_index,
        completed_episodes,
    ) in enumerate(
        range(
            START_EPISODES
            + EPISODES_PER_UPDATE,
            total_episodes + 1,
            EPISODES_PER_UPDATE,
        )
    ):
        rows.append(
            run_episode_batch(
                completed_episodes,
                update_index,
            )
        )

        display(
            pd.DataFrame(
                rows
            )
        )

    return pd.DataFrame(
        rows
    )


## ループ実行

このセルを実行すると、最新の `model_episode...` から継続し、3600エピソードの対戦→学習を繰り返します。

学習率は各3600エピソード学習の内部では固定し、その更新が終わるたびにCosine schedulerを1 step進めます。したがって200回の更新全体で `1e-4 → 1e-5` と滑らかに低下します。AdamWの1次・2次モーメント、optimizer step、scheduler stateもすべて継続します。


In [20]:
results = run_training_loop()
print(f'completed: {RUN_ROOT}')

parallel games: workers=15, lanes/worker=400, games=6000, pairings=136, self=yes, limits=100 turns/500 selections
balanced schedule: matches/agent=706〜706, player-sides/agent=750〜750


CalledProcessError: Command '['C:\\mizuki\\PokeTCG\\pokemon-tcg-agent\\.venv\\Scripts\\python.exe', 'C:\\mizuki\\PokeTCG\\pokemon-tcg-agent\\tools\\run_matches_round_robin.py', '--agent', 'cluster_00=C:\\mizuki\\PokeTCG\\pokemon-tcg-agent\\agents\\16model_pretrained_vtrain\\cluster_00\\src\\main.py', '--agent', 'cluster_01=C:\\mizuki\\PokeTCG\\pokemon-tcg-agent\\agents\\16model_pretrained_vtrain\\cluster_01\\src\\main.py', '--agent', 'cluster_02=C:\\mizuki\\PokeTCG\\pokemon-tcg-agent\\agents\\16model_pretrained_vtrain\\cluster_02\\src\\main.py', '--agent', 'cluster_03=C:\\mizuki\\PokeTCG\\pokemon-tcg-agent\\agents\\16model_pretrained_vtrain\\cluster_03\\src\\main.py', '--agent', 'cluster_04=C:\\mizuki\\PokeTCG\\pokemon-tcg-agent\\agents\\16model_pretrained_vtrain\\cluster_04\\src\\main.py', '--agent', 'cluster_05=C:\\mizuki\\PokeTCG\\pokemon-tcg-agent\\agents\\16model_pretrained_vtrain\\cluster_05\\src\\main.py', '--agent', 'cluster_06=C:\\mizuki\\PokeTCG\\pokemon-tcg-agent\\agents\\16